# Empath-IQ: Full Fusion Training Notebook
**Runtime → Change runtime type → T4 GPU** before running.

## What this notebook does
1. Fine-tunes `google/vit-base-patch16-224` on RAF-DB → frozen emotion encoder
2. Defines posture feature extraction (same logic as `app.py`)  
3. Generates posture+behavior training data (synthetic + easy to extend with real data)
4. Trains a tiny Fusion MLP: `[7 emotion probs + 8 posture features] → 10 behaviors`
5. Exports `emotion_vit.pt` + `fusion_mlp.pt` + `behavior_labels.json`

**Training time:** ~20 min total on T4

In [ ]:
!pip install transformers datasets timm Pillow tqdm scikit-learn -q
print('✅ Done')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from datasets import load_dataset
from PIL import Image
import numpy as np
import math, json, os
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Config ──────────────────────────────────────────────────────
MODEL_NAME  = 'google/vit-base-patch16-224'
IMG_SIZE    = 224
BATCH_SIZE  = 32
VIT_EPOCHS  = 5
MLP_EPOCHS  = 80
VIT_LR      = 2e-4
MLP_LR      = 1e-3

# RAF-DB label order: 0=Surprise,1=Fear,2=Disgust,3=Happy,4=Sad,5=Angry,6=Neutral
EMOTION_LABELS = ['surprise', 'fear', 'disgust', 'happy', 'sad', 'angry', 'neutral']

BEHAVIOR_LABELS = [
    'Engaged & Attentive',
    'Stressed / Anxious',
    'Disengaged / Bored',
    'Confident & Expressive',
    'Defensive',
    'Excited / Animated',
    'Sad / Withdrawn',
    'Thinking / Reflective',
    'Frustrated',
    'Calm & Relaxed',
]

N_EMO      = len(EMOTION_LABELS)   # 7
N_POSTURE  = 8                     # posture feature vector size
N_BEHAVIOR = len(BEHAVIOR_LABELS)  # 10

print(f'Emotions:  {EMOTION_LABELS}')
print(f'Behaviors: {BEHAVIOR_LABELS}')

In [ ]:
# ── Load RAF-DB ──────────────────────────────────────────────────
print('Loading RAF-DB...')
ds = load_dataset('rrathna/RAF-DB', trust_remote_code=True)
print(ds)
print(f"Train: {len(ds['train'])}  Test: {len(ds['test'])}")

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

class RAFDBDataset(Dataset):
    def __init__(self, hf_ds, tf):
        self.data = hf_ds
        self.tf   = tf
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img  = item['image']
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        return self.tf(img.convert('RGB')), item['label']

train_loader = DataLoader(RAFDBDataset(ds['train'], train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(RAFDBDataset(ds['test'],  val_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
# ── Build ViT ────────────────────────────────────────────────────
vit = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_EMO,
    ignore_mismatched_sizes=True,
    id2label={i: l for i, l in enumerate(EMOTION_LABELS)},
    label2id={l: i for i, l in enumerate(EMOTION_LABELS)},
).to(DEVICE)

# Freeze everything
for p in vit.parameters(): p.requires_grad = False

# Unfreeze: classifier head + last 2 transformer blocks + final layernorm
for p in vit.classifier.parameters():       p.requires_grad = True
for p in vit.vit.layernorm.parameters():    p.requires_grad = True
for block in vit.vit.encoder.layer[-2:]:
    for p in block.parameters():            p.requires_grad = True

trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
total     = sum(p.numel() for p in vit.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

In [ ]:
# ── Train ViT ────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, vit.parameters()),
                        lr=VIT_LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=VIT_EPOCHS)

def run_epoch(model, loader, opt=None):
    model.train() if opt else model.eval()
    loss_sum, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if opt else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if opt: opt.zero_grad()
            out  = model(pixel_values=imgs)
            loss = criterion(out.logits, labels)
            if opt:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            loss_sum += loss.item()
            correct  += (out.logits.argmax(1) == labels).sum().item()
            total    += labels.size(0)
    return loss_sum / len(loader), correct / total

best_vit_acc = 0
vit_history  = []
for epoch in range(1, VIT_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(vit, train_loader, optimizer)
    vl_loss, vl_acc = run_epoch(vit, val_loader)
    scheduler.step()
    vit_history.append((epoch, tr_loss, tr_acc, vl_loss, vl_acc))
    print(f'Epoch {epoch}/{VIT_EPOCHS}  train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}'
          f'  val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}')
    if vl_acc > best_vit_acc:
        best_vit_acc = vl_acc
        torch.save(vit.state_dict(), 'vit_best.pt')
        print(f'  ✅ saved (val_acc={vl_acc:.4f})')

vit.load_state_dict(torch.load('vit_best.pt', map_location=DEVICE))
vit.eval()
print(f'Best ViT val acc: {best_vit_acc:.4f}')

In [ ]:
# ── Posture feature extraction (mirrors app.py exactly) ─────────
def posture_features_from_vec(v):
    """
    v: list/array of 8 floats in order:
       [shoulder_tilt, head_forward, spine_angle, shoulder_width,
        arm_raise, head_tilt, openness, slouch_score]
    Returns: posture_label (str)
    """
    shoulder_tilt, head_forward, spine_angle, shoulder_width, \
        arm_raise, head_tilt, openness, slouch_score = v

    if spine_angle > 15:      return 'leaning'
    elif slouch_score > 0.15: return 'slouched'
    elif arm_raise > 0.1:     return 'arms_raised'
    elif openness > 1.2:      return 'open'
    elif openness < 0.6:      return 'closed'
    else:                     return 'neutral'

POSTURE_LABELS = ['neutral', 'slouched', 'leaning', 'open', 'closed', 'arms_raised']
print('Posture feature order:')
print('  [shoulder_tilt, head_forward, spine_angle, shoulder_width,')
print('   arm_raise, head_tilt, openness, slouch_score]')

In [ ]:
# ── Generate Fusion Training Data ───────────────────────────────
# Each sample: 7 emotion probs + 8 posture features → behavior label
# We generate samples consistent with each behavior's definition.
# The more samples, the better — 500/class is plenty for an MLP.

np.random.seed(42)
N_PER_CLASS = 500

EMO_IDX = {e: i for i, e in enumerate(EMOTION_LABELS)}

def emo_vec(dominant, strength=0.75):
    v = np.random.dirichlet(np.ones(N_EMO) * 0.3)
    v[EMO_IDX[dominant]] = strength + np.random.rand() * (1 - strength) * 0.5
    v /= v.sum()
    return v

def pose_vec(shoulder_tilt, spine_angle, arm_raise, head_tilt, openness, slouch, noise=0.02):
    base = np.array([
        shoulder_tilt,          # shoulder_tilt
        np.random.randn()*0.01, # head_forward (minor)
        spine_angle,            # spine_angle
        0.28 + np.random.randn()*0.02,  # shoulder_width
        arm_raise,              # arm_raise
        head_tilt,              # head_tilt
        openness,               # openness
        slouch,                 # slouch_score
    ], dtype=np.float32)
    return base + np.random.randn(N_POSTURE).astype(np.float32) * noise

# Behavior templates: (dominant_emo, posture params)
# posture params: (shoulder_tilt, spine_angle, arm_raise, head_tilt, openness, slouch)
TEMPLATES = {
    'Engaged & Attentive':    [('happy',    0.02, 5,  0.02, 0.02, 0.90, 0.04),
                                ('surprise', 0.02, 4,  0.02, 0.02, 0.95, 0.03)],
    'Stressed / Anxious':     [('fear',     0.04, 8,  0.01, 0.02, 0.50, 0.10),
                                ('angry',    0.04, 7,  0.01, 0.03, 0.48, 0.12)],
    'Disengaged / Bored':     [('neutral',  0.03, 5,  0.01, 0.02, 0.80, 0.20),
                                ('neutral',  0.04,12,  0.01, 0.02, 0.85, 0.22)],
    'Confident & Expressive': [('happy',    0.02, 4,  0.03, 0.03, 1.40, 0.04),
                                ('neutral',  0.02, 4,  0.02, 0.02, 1.35, 0.04)],
    'Defensive':              [('angry',    0.03, 6,  0.01, 0.02, 0.42, 0.08),
                                ('fear',     0.03, 6,  0.01, 0.02, 0.45, 0.09)],
    'Excited / Animated':     [('surprise', 0.04, 5,  0.15, 0.03, 1.10, 0.05),
                                ('happy',    0.04, 5,  0.14, 0.03, 1.15, 0.04)],
    'Sad / Withdrawn':        [('sad',      0.03, 5,  0.01, 0.02, 0.55, 0.18),
                                ('disgust',  0.03, 5,  0.01, 0.02, 0.52, 0.20)],
    'Thinking / Reflective':  [('neutral',  0.02, 5,  0.01, 0.07, 0.85, 0.06),
                                ('neutral',  0.02, 5,  0.01, 0.09, 0.80, 0.05)],
    'Frustrated':             [('angry',    0.04,18,  0.02, 0.03, 1.10, 0.08),
                                ('disgust',  0.04,16,  0.02, 0.03, 1.05, 0.09)],
    'Calm & Relaxed':         [('neutral',  0.02, 4,  0.02, 0.02, 0.90, 0.06),
                                ('happy',    0.02, 4,  0.02, 0.02, 0.92, 0.05)],
}

X_list, y_list = [], []

for beh_idx, beh_label in enumerate(BEHAVIOR_LABELS):
    templates = TEMPLATES[beh_label]
    per_template = N_PER_CLASS // len(templates)
    for tmpl in templates:
        dom_emo, sh_tilt, spine_ang, arm_raise, head_tilt, openness, slouch = tmpl
        for _ in range(per_template):
            ev = emo_vec(dom_emo)
            pv = pose_vec(sh_tilt, spine_ang, arm_raise, head_tilt, openness, slouch)
            X_list.append(np.concatenate([ev, pv]))
            y_list.append(beh_idx)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
print(f'Fusion dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Class distribution: {np.bincount(y)}')

In [ ]:
# ── Fusion MLP ───────────────────────────────────────────────────
class FusionMLP(nn.Module):
    """
    Input:  15-dim vector [7 emotion probs | 8 posture features]
    Output: 10-dim logits over behavior classes
    Tiny but effective — ~5k parameters.
    """
    def __init__(self, in_dim=15, hidden=64, out_dim=10, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

mlp = FusionMLP(in_dim=N_EMO + N_POSTURE, hidden=64, out_dim=N_BEHAVIOR).to(DEVICE)
total_params = sum(p.numel() for p in mlp.parameters())
print(f'Fusion MLP parameters: {total_params:,}')

In [ ]:
# ── Train Fusion MLP ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_tr, X_vl, y_tr, y_vl = train_test_split(X, y, test_size=0.15,
                                            stratify=y, random_state=42)

def make_loader(Xd, yd, shuffle=True):
    ds = TensorDataset(torch.tensor(Xd), torch.tensor(yd))
    return DataLoader(ds, batch_size=128, shuffle=shuffle)

mlp_train_loader = make_loader(X_tr, y_tr)
mlp_val_loader   = make_loader(X_vl, y_vl, shuffle=False)

mlp_criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
mlp_optimizer = optim.AdamW(mlp.parameters(), lr=MLP_LR, weight_decay=1e-4)
mlp_scheduler = optim.lr_scheduler.CosineAnnealingLR(mlp_optimizer, T_max=MLP_EPOCHS)

best_mlp_acc = 0
mlp_history  = []

for epoch in range(1, MLP_EPOCHS + 1):
    # Train
    mlp.train()
    tr_loss, tr_correct, tr_total = 0, 0, 0
    for xb, yb in mlp_train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        mlp_optimizer.zero_grad()
        out  = mlp(xb)
        loss = mlp_criterion(out, yb)
        loss.backward()
        mlp_optimizer.step()
        tr_loss    += loss.item()
        tr_correct += (out.argmax(1) == yb).sum().item()
        tr_total   += yb.size(0)
    mlp_scheduler.step()

    # Val
    mlp.eval()
    vl_correct, vl_total = 0, 0
    with torch.no_grad():
        for xb, yb in mlp_val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl_correct += (mlp(xb).argmax(1) == yb).sum().item()
            vl_total   += yb.size(0)

    tr_acc = tr_correct / tr_total
    vl_acc = vl_correct / vl_total
    mlp_history.append((epoch, tr_loss/len(mlp_train_loader), tr_acc, vl_acc))

    if vl_acc > best_mlp_acc:
        best_mlp_acc = vl_acc
        torch.save(mlp.state_dict(), 'fusion_mlp_best.pt')

    if epoch % 10 == 0 or epoch == MLP_EPOCHS:
        print(f'Epoch {epoch:3d}/{MLP_EPOCHS}  '
              f'train_acc={tr_acc:.4f}  val_acc={vl_acc:.4f}')

mlp.load_state_dict(torch.load('fusion_mlp_best.pt', map_location=DEVICE))
mlp.eval()
print(f'\nBest Fusion MLP val acc: {best_mlp_acc:.4f}')

In [ ]:
# ── Evaluate ─────────────────────────────────────────────────────
mlp.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in mlp_val_loader:
        preds = mlp(xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

print(classification_report(all_true, all_preds,
                             target_names=BEHAVIOR_LABELS, digits=3))

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
short_labels = [b.split(' & ')[0].split(' / ')[0] for b in BEHAVIOR_LABELS]
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_labels, yticklabels=short_labels)
plt.title('Fusion MLP — Behavior Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('behavior_confusion.png', dpi=120)
plt.show()

In [ ]:
# ── Export both models ───────────────────────────────────────────
# 1. ViT checkpoint
torch.save({
    'state_dict':    vit.state_dict(),
    'labels':        EMOTION_LABELS,
    'model_name':    MODEL_NAME,
    'img_size':      IMG_SIZE,
    'best_val_acc':  best_vit_acc,
}, 'emotion_vit.pt')

# 2. Fusion MLP checkpoint
torch.save({
    'state_dict':      mlp.state_dict(),
    'behavior_labels': BEHAVIOR_LABELS,
    'emotion_labels':  EMOTION_LABELS,
    'in_dim':          N_EMO + N_POSTURE,
    'hidden':          64,
    'out_dim':         N_BEHAVIOR,
    'best_val_acc':    best_mlp_acc,
}, 'fusion_mlp.pt')

# 3. Label files (for app.py)
with open('behavior_labels.json', 'w') as f:
    json.dump(BEHAVIOR_LABELS, f)
with open('emotion_labels.json', 'w') as f:
    json.dump(EMOTION_LABELS, f)

print('✅ Exported:')
print('   emotion_vit.pt    — ViT emotion encoder')
print('   fusion_mlp.pt     — behavior fusion model')
print('   behavior_labels.json')
print('   emotion_labels.json')

In [ ]:
# ── Training curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ViT accuracy
ep_v = [h[0] for h in vit_history]
axes[0].plot(ep_v, [h[2] for h in vit_history], label='Train')
axes[0].plot(ep_v, [h[4] for h in vit_history], label='Val')
axes[0].set_title('ViT Emotion — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].legend()

# MLP accuracy
ep_m = [h[0] for h in mlp_history]
axes[1].plot(ep_m, [h[2] for h in mlp_history], label='Train')
axes[1].plot(ep_m, [h[3] for h in mlp_history], label='Val')
axes[1].set_title('Fusion MLP — Accuracy')
axes[1].set_xlabel('Epoch'); axes[1].legend()

# MLP loss
axes[2].plot(ep_m, [h[1] for h in mlp_history], label='Train loss')
axes[2].set_title('Fusion MLP — Loss')
axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Quick inference test ─────────────────────────────────────────
# Simulate what app.py does at runtime

def infer_behavior(emotion_probs: np.ndarray,
                   posture_features: np.ndarray) -> tuple:
    """
    emotion_probs:    shape (7,)  — softmax output from ViT (0-1)
    posture_features: shape (8,)  — from MediaPipe landmarks
    Returns: (behavior_label, confidence, all_probs dict)
    """
    x = np.concatenate([emotion_probs, posture_features]).astype(np.float32)
    t = torch.tensor(x).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = mlp(t)
        probs  = F.softmax(logits, dim=-1)[0].cpu().numpy()
    idx       = int(np.argmax(probs))
    label     = BEHAVIOR_LABELS[idx]
    confidence = float(probs[idx])
    all_probs  = {BEHAVIOR_LABELS[i]: round(float(p)*100, 1) for i, p in enumerate(probs)}
    return label, confidence, all_probs

# Test: happy emotion + open posture → should be Engaged/Confident
test_emo  = np.array([0.05, 0.02, 0.02, 0.82, 0.03, 0.03, 0.03])  # dominant=happy
test_pose = np.array([0.02, 0.0, 4.5, 0.30, 0.02, 0.02, 1.35, 0.05])  # open posture

label, conf, all_probs = infer_behavior(test_emo, test_pose)
print(f'Test prediction: "{label}" (confidence: {conf:.2%})')
print('All probabilities:')
for b, p in sorted(all_probs.items(), key=lambda x: -x[1]):
    bar = '█' * int(p / 5)
    print(f'  {b:<30} {p:5.1f}%  {bar}')

In [ ]:
# ── Download ─────────────────────────────────────────────────────
from google.colab import files
files.download('emotion_vit.pt')
files.download('fusion_mlp.pt')
files.download('behavior_labels.json')
files.download('emotion_labels.json')
files.download('training_curves.png')
files.download('behavior_confusion.png')
print('Done! Place emotion_vit.pt and fusion_mlp.pt in /repo/empath-iq/')

## Integration with app.py

After downloading, place both `.pt` files in `/repo/empath-iq/` and use the updated `app.py` provided alongside this notebook. The key change in `app.py` is:

**Old flow (rule-based behavior):**
```
ViT → dominant_emotion → rule lookup → behavior
```

**New flow (learned fusion):**
```
ViT → emotion_probs (7 floats) ──┐
                                   ├→ FusionMLP → behavior label + confidence
MediaPipe → posture_features ─────┘
```

The `infer_behavior()` function in `app.py` is replaced with a single MLP forward pass.